# PyTorch Image Classification Notebook
A simple, customizable notebook for image classification experiments with PyTorch.

**Features:**
- Easy model customization and experimentation
- Transfer learning with pretrained models
- Training with validation tracking
- Confusion matrix and error analysis
- Feature extraction for embeddings
- Inference on test data

## 1. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
from pathlib import Path
from PIL import Image
import os

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Configuration
Set your paths and hyperparameters here

In [ ]:
# === PATHS ===
DATA_DIR = './data'  # Change to your data directory
TRAIN_DIR = os.path.join(DATA_DIR, 'train')  # Expected structure: train/class1/, train/class2/, ...
TEST_DIR = os.path.join(DATA_DIR, 'test')    # Optional: for unlabeled test images

# === HYPERPARAMETERS ===
IMAGE_SIZE = 224
BATCH_SIZE = 32
LEARNING_RATE = 0.001
NUM_EPOCHS = 10
VAL_SPLIT = 0.2  # Validation split ratio

# === MODEL SETTINGS ===
MODEL_NAME = 'resnet18'  # Options: resnet18, resnet50, efficientnet_b0, mobilenet_v2, vit_b_16
SAVE_PATH = './best_model.pth'

## 3. Data Preparation

In [ ]:
# === DATA TRANSFORMS ===
# Training: with augmentation
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Validation/Test: no augmentation
val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
# === LOAD DATA ===
full_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
class_names = full_dataset.classes
num_classes = len(class_names)

print(f"Classes ({num_classes}): {class_names}")
print(f"Total samples: {len(full_dataset)}")

# Split into train and validation
val_size = int(VAL_SPLIT * len(full_dataset))
train_size = len(full_dataset) - val_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Apply validation transforms to val_dataset
val_dataset.dataset = datasets.ImageFolder(TRAIN_DIR, transform=val_transforms)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Training samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")

### Visualize Sample Data

In [ ]:
def imshow(images, labels, num_samples=8):
    """Display a batch of images with labels"""
    fig, axes = plt.subplots(1, num_samples, figsize=(15, 3))
    for idx in range(min(num_samples, len(images))):
        img = images[idx].cpu().numpy().transpose((1, 2, 0))
        # Denormalize
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img = std * img + mean
        img = np.clip(img, 0, 1)
        
        axes[idx].imshow(img)
        axes[idx].set_title(class_names[labels[idx]])
        axes[idx].axis('off')
    plt.tight_layout()
    plt.show()

# Show sample batch
sample_images, sample_labels = next(iter(train_loader))
imshow(sample_images, sample_labels)

## 4. Model Definition
Easily swap models by changing MODEL_NAME in configuration

In [ ]:
def create_model(model_name, num_classes, pretrained=True):
    """
    Create a model with a custom classifier head
    
    Supported models: resnet18, resnet50, efficientnet_b0, mobilenet_v2, vit_b_16
    """
    
    if model_name == 'resnet18':
        model = models.resnet18(pretrained=pretrained)
        num_features = model.fc.in_features
        model.fc = nn.Linear(num_features, num_classes)
        
    elif model_name == 'resnet50':
        model = models.resnet50(pretrained=pretrained)
        num_features = model.fc.in_features
        model.fc = nn.Linear(num_features, num_classes)
        
    elif model_name == 'efficientnet_b0':
        model = models.efficientnet_b0(pretrained=pretrained)
        num_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(num_features, num_classes)
        
    elif model_name == 'mobilenet_v2':
        model = models.mobilenet_v2(pretrained=pretrained)
        num_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(num_features, num_classes)
        
    elif model_name == 'vit_b_16':
        model = models.vit_b_16(pretrained=pretrained)
        num_features = model.heads.head.in_features
        model.heads.head = nn.Linear(num_features, num_classes)
        
    else:
        raise ValueError(f"Model {model_name} not supported")
    
    return model

# Create model
model = create_model(MODEL_NAME, num_classes, pretrained=True)
model = model.to(device)

print(f"\nModel: {MODEL_NAME}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

### Optional: Freeze Base Layers (for faster training)
Unfreeze them later for fine-tuning

In [ ]:
def freeze_base_layers(model, model_name):
    """Freeze all layers except the classifier head"""
    for param in model.parameters():
        param.requires_grad = False
    
    # Unfreeze classifier head based on model architecture
    if 'resnet' in model_name:
        for param in model.fc.parameters():
            param.requires_grad = True
    elif 'efficientnet' in model_name or 'mobilenet' in model_name:
        for param in model.classifier.parameters():
            param.requires_grad = True
    elif 'vit' in model_name:
        for param in model.heads.parameters():
            param.requires_grad = True

# Uncomment to freeze base layers
# freeze_base_layers(model, MODEL_NAME)
# print(f"Trainable parameters after freezing: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 5. Training

In [ ]:
# Define loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Optional: Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5, verbose=True)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


def validate(model, loader, criterion, device):
    """Validate the model"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc, all_preds, all_labels

In [ ]:
# === TRAINING LOOP ===
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0

print("Starting training...\n")

for epoch in range(NUM_EPOCHS):
    # Train
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_loss, val_acc, _, _ = validate(model, val_loader, criterion, device)
    
    # Update scheduler
    scheduler.step(val_loss)
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    # Print progress
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"  Val Loss:   {val_loss:.4f}, Val Acc:   {val_acc:.4f}")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  ✓ Saved best model (val_acc: {val_acc:.4f})")
    print()

print(f"Training complete! Best validation accuracy: {best_val_acc:.4f}")

### Plot Training History

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss
ax1.plot(history['train_loss'], label='Train Loss')
ax1.plot(history['val_loss'], label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True)

# Accuracy
ax2.plot(history['train_acc'], label='Train Acc')
ax2.plot(history['val_acc'], label='Val Acc')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## 6. Evaluation & Metrics

In [ ]:
# Load best model
model.load_state_dict(torch.load(SAVE_PATH))
print(f"Loaded best model from {SAVE_PATH}")

# Get predictions
_, val_acc, predictions, true_labels = validate(model, val_loader, criterion, device)
print(f"\nValidation Accuracy: {val_acc:.4f}")

### Classification Report

In [ ]:
print("\nClassification Report:")
print("=" * 60)
print(classification_report(true_labels, predictions, target_names=class_names))

### Confusion Matrix

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(true_labels, predictions)

# Plot
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

# Print per-class accuracy
print("\nPer-class Accuracy:")
for i, class_name in enumerate(class_names):
    class_acc = cm[i, i] / cm[i].sum() if cm[i].sum() > 0 else 0
    print(f"  {class_name}: {class_acc:.4f}")

## 7. Error Analysis
Identify misclassified samples to understand model weaknesses

In [ ]:
def analyze_errors(model, loader, class_names, device, num_samples=10):
    """
    Show misclassified samples
    """
    model.eval()
    misclassified = []
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)
            
            # Find misclassified
            for i in range(len(labels)):
                if predicted[i] != labels[i]:
                    misclassified.append({
                        'image': images[i].cpu(),
                        'true_label': labels[i].cpu().item(),
                        'pred_label': predicted[i].cpu().item(),
                        'confidence': probs[i][predicted[i]].cpu().item(),
                        'true_prob': probs[i][labels[i]].cpu().item()
                    })
    
    # Plot misclassified samples
    if len(misclassified) == 0:
        print("No misclassifications found!")
        return
    
    num_to_show = min(num_samples, len(misclassified))
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    axes = axes.flatten()
    
    for idx in range(num_to_show):
        error = misclassified[idx]
        img = error['image'].numpy().transpose((1, 2, 0))
        
        # Denormalize
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img = std * img + mean
        img = np.clip(img, 0, 1)
        
        axes[idx].imshow(img)
        true_name = class_names[error['true_label']]
        pred_name = class_names[error['pred_label']]
        axes[idx].set_title(f"True: {true_name}\nPred: {pred_name}\nConf: {error['confidence']:.2f}", 
                           fontsize=9)
        axes[idx].axis('off')
    
    # Hide unused subplots
    for idx in range(num_to_show, len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle(f"Misclassified Samples (Total: {len(misclassified)})", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f"\nTotal misclassifications: {len(misclassified)}")
    return misclassified

# Analyze errors
errors = analyze_errors(model, val_loader, class_names, device, num_samples=10)

### Most Confused Classes

In [ ]:
def show_confused_pairs(cm, class_names, top_n=5):
    """Show which class pairs are most confused"""
    confused_pairs = []
    
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            if i != j and cm[i, j] > 0:
                confused_pairs.append({
                    'true_class': class_names[i],
                    'pred_class': class_names[j],
                    'count': cm[i, j]
                })
    
    # Sort by count
    confused_pairs = sorted(confused_pairs, key=lambda x: x['count'], reverse=True)
    
    print(f"\nTop {top_n} Confused Class Pairs:")
    print("=" * 60)
    for idx, pair in enumerate(confused_pairs[:top_n], 1):
        print(f"{idx}. True: {pair['true_class']:15s} → Predicted as: {pair['pred_class']:15s} ({pair['count']} times)")

show_confused_pairs(cm, class_names)

## 8. Feature Extraction / Embeddings
Extract features from the model for clustering, similarity search, etc.

In [ ]:
def get_feature_extractor(model, model_name):
    """
    Create a feature extractor by removing the classifier head
    """
    if 'resnet' in model_name:
        # Remove final FC layer
        feature_extractor = nn.Sequential(*list(model.children())[:-1])
    elif 'efficientnet' in model_name or 'mobilenet' in model_name:
        # Remove classifier
        feature_extractor = nn.Sequential(model.features, model.avgpool)
    elif 'vit' in model_name:
        # For ViT, we'll extract before the head
        class ViTFeatureExtractor(nn.Module):
            def __init__(self, vit_model):
                super().__init__()
                self.vit = vit_model
                
            def forward(self, x):
                x = self.vit._process_input(x)
                n = x.shape[0]
                batch_class_token = self.vit.class_token.expand(n, -1, -1)
                x = torch.cat([batch_class_token, x], dim=1)
                x = self.vit.encoder(x)
                return x[:, 0]  # Return class token
        
        feature_extractor = ViTFeatureExtractor(model)
    else:
        raise ValueError(f"Feature extraction not implemented for {model_name}")
    
    return feature_extractor

# Create feature extractor
feature_extractor = get_feature_extractor(model, MODEL_NAME)
feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

print("Feature extractor created")

In [ ]:
def extract_features(model, loader, device):
    """Extract features for all samples in the loader"""
    features = []
    labels = []
    
    model.eval()
    with torch.no_grad():
        for images, lbls in loader:
            images = images.to(device)
            feats = model(images)
            feats = feats.view(feats.size(0), -1)  # Flatten
            
            features.append(feats.cpu().numpy())
            labels.append(lbls.numpy())
    
    features = np.concatenate(features, axis=0)
    labels = np.concatenate(labels, axis=0)
    
    return features, labels

# Extract features from validation set
print("Extracting features from validation set...")
val_features, val_labels = extract_features(feature_extractor, val_loader, device)
print(f"Extracted features shape: {val_features.shape}")
print(f"Feature dimension: {val_features.shape[1]}")

### Visualize Features with t-SNE

In [ ]:
from sklearn.manifold import TSNE

# Apply t-SNE (sample if dataset is large)
max_samples = 1000
if len(val_features) > max_samples:
    indices = np.random.choice(len(val_features), max_samples, replace=False)
    features_tsne = val_features[indices]
    labels_tsne = val_labels[indices]
else:
    features_tsne = val_features
    labels_tsne = val_labels

print(f"Running t-SNE on {len(features_tsne)} samples...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
features_2d = tsne.fit_transform(features_tsne)

# Plot
plt.figure(figsize=(10, 8))
for i, class_name in enumerate(class_names):
    mask = labels_tsne == i
    plt.scatter(features_2d[mask, 0], features_2d[mask, 1], 
               label=class_name, alpha=0.6, s=30)

plt.legend()
plt.title('t-SNE Visualization of Feature Space')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Inference on Test Data
For unlabeled test images

In [ ]:
def predict_test_images(model, test_dir, transform, class_names, device, save_csv=True):
    """
    Predict labels for unlabeled test images
    """
    if not os.path.exists(test_dir):
        print(f"Test directory not found: {test_dir}")
        return None
    
    model.eval()
    results = []
    
    # Get all image files
    valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp')
    image_files = [f for f in os.listdir(test_dir) if f.lower().endswith(valid_extensions)]
    
    if len(image_files) == 0:
        print(f"No images found in {test_dir}")
        return None
    
    print(f"Processing {len(image_files)} test images...")
    
    with torch.no_grad():
        for img_name in image_files:
            img_path = os.path.join(test_dir, img_name)
            
            try:
                # Load and transform image
                image = Image.open(img_path).convert('RGB')
                image_tensor = transform(image).unsqueeze(0).to(device)
                
                # Predict
                output = model(image_tensor)
                probabilities = torch.softmax(output, dim=1)[0]
                pred_class = torch.argmax(probabilities).item()
                confidence = probabilities[pred_class].item()
                
                results.append({
                    'filename': img_name,
                    'predicted_class': class_names[pred_class],
                    'predicted_label': pred_class,
                    'confidence': confidence
                })
                
            except Exception as e:
                print(f"Error processing {img_name}: {e}")
    
    # Create DataFrame
    df = pd.DataFrame(results)
    
    if save_csv and len(df) > 0:
        csv_path = os.path.join(DATA_DIR, 'predictions.csv')
        df.to_csv(csv_path, index=False)
        print(f"\nSaved predictions to {csv_path}")
    
    return df

# Run inference on test directory
if os.path.exists(TEST_DIR):
    predictions_df = predict_test_images(model, TEST_DIR, val_transforms, class_names, device)
    if predictions_df is not None:
        print("\nSample predictions:")
        print(predictions_df.head(10))
else:
    print(f"Test directory not found: {TEST_DIR}")
    print("To use inference, create a test directory with images")

## 10. Save & Load Model

In [ ]:
# The best model is already saved during training
# To load it later:

# model = create_model(MODEL_NAME, num_classes, pretrained=False)
# model.load_state_dict(torch.load(SAVE_PATH))
# model = model.to(device)
# model.eval()

print(f"✓ Model saved at: {SAVE_PATH}")
print(f"✓ To load: model.load_state_dict(torch.load('{SAVE_PATH}'))")

## Summary

This notebook provides a simple, customizable workflow for image classification:

1. **Easy Configuration**: Change model, hyperparameters, and paths in one place
2. **Data Augmentation**: Built-in transforms for better generalization
3. **Multiple Models**: Easy to switch between ResNet, EfficientNet, MobileNet, ViT
4. **Training Loop**: Clear, simple training with validation tracking
5. **Evaluation**: Confusion matrix, classification report, per-class metrics
6. **Error Analysis**: Visualize misclassifications and confused class pairs
7. **Feature Extraction**: Get embeddings for downstream tasks
8. **Inference**: Predict on new unlabeled images

### Quick Tips:
- **Change model**: Modify `MODEL_NAME` in configuration
- **Faster training**: Freeze base layers with `freeze_base_layers()`
- **Better performance**: Increase `NUM_EPOCHS`, tune `LEARNING_RATE`
- **More augmentation**: Add transforms like `RandomCrop`, `RandomAffine`
- **Fine-tuning**: Train with frozen layers first, then unfreeze and continue training